# Create a custom algorithm in the notebook

This example defines a tiny validation algorithm directly in the notebook, inserts it into a pipeline, and runs one event through it.


## Bootstrap project imports


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate reco_algorithm_tests project root.")

src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Project root: {PROJECT_ROOT}")


Project root: /workdir/playground/reco_algorithm_tests


## Import reconstruction tools


In [2]:
from algorithms import DefaultTrackletFormer, EventValidator
from data_io import RecoDataFile, load_pioneer_libraries
from pipeline.pipeline import Pipeline
from pipeline.stages import EventInitStage, EventInputStage, InputContext, TrackletStage, ValidationStage


## Load PIONEER libraries and the smoke-test data


In [3]:
load_pioneer_libraries()

DATA_PATH = PROJECT_ROOT / ".data" / "current" / "all_rec.root"
ENTRY_INDEX = 0

reco_data = RecoDataFile(str(DATA_PATH))
print(f"Loaded data file: {DATA_PATH}")
print(f"Entries available: {reco_data.entries}")


Loaded data file: /workdir/playground/reco_algorithm_tests/.data/current/all_rec.root
Entries available: 727


## Define a new algorithm inline

This validator is intentionally simple: it marks an event as valid if it contains at least a chosen number of reconstructed tracklets.


In [4]:
class MinimumTrackletValidator(EventValidator):
    def __init__(self, min_tracklets: int = 2):
        self.min_tracklets = int(min_tracklets)

    def validate(self, event, storage=None) -> bool:
        n_tracklets = len(event.all_tracklets)
        if storage is not None:
            storage.setdefault("extra_info", {})["custom_validation"] = {
                "n_tracklets": n_tracklets,
                "min_tracklets": self.min_tracklets,
            }
        return n_tracklets >= self.min_tracklets


## Insert the algorithm into a pipeline


In [5]:
pipeline = Pipeline()
pipeline.register_stage(EventInputStage())
pipeline.register_stage(EventInitStage())
pipeline.register_stage(TrackletStage(DefaultTrackletFormer()))
pipeline.register_stage(ValidationStage(MinimumTrackletValidator(min_tracklets=2)))

pipeline.run(InputContext(reco_data, ENTRY_INDEX))
event = pipeline.get_event()
event


Event(id=0, no patterns, 2 tracklets (sample IDs: [0, 1]), no vertices, 33 hits, extra_info: 3 keys: ['geo', 'tracklet_algorithm_info', 'custom_validation'], is_valid=True)

## Inspect the result


In [6]:
print("Event valid:", event.is_valid)
print("Tracklets in event:", len(event.all_tracklets))
print("Stored algorithm info:", pipeline.get_storage()["extra_info"].get("custom_validation"))
pipeline.status()


Event valid: True
Tracklets in event: 2
Stored algorithm info: {'n_tracklets': 2, 'min_tracklets': 2}
Pipeline Status (generated in 0.0000s):
  Total stages: 4
  Completed: 4
  Current stage: validation
  Next stage: None
  All completed: True
  Storage keys: ['event_data', 'truth_data', 'geo', 'extra_info', 'entry_index', 'event_entry', 'raw_hits', 'reference_truth_entry', 'patterns', 'tracklets', 'vertices', 'event']

Stage Details:
Load Event Input Data: done  runnable
Initialize Event: done  runnable
Form Tracklets: done  runnable
Validate Event: done current runnable
